# 🧬 00 · Welcome: from a crowded cell to molecular structures
### What cryo-EM and cryo-ET can see, why computation is the bottleneck, and how this course is organised

**BiologyToResearch** · Part A — Foundations · Module 00 of 34

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/cryoem/tutorial/BiologyToResearch/notebooks/00_welcome_cells_to_structures.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy, Matplotlib |
| **Prerequisites** | Introductory cell biology. **No programming, physics or calculus needed.** |
| **Estimated time** | 60–75 min (reading + running + the ✏️ exercises) |
| **Lab papers covered** | Beck *et al.* 2011 (visual proteomics); Kim *et al.* 2023 (review of cryo-ET pattern mining); AITom (Zeng & Xu 2019; Zhan *et al.* 2025); book chapters Han & Xu 2024, Uddin *et al.* 2025 — see [`PAPER_ATLAS.md`](../PAPER_ATLAS.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> All experiments in this course are small, controlled *toy versions* of published methods. They are designed to teach
> the core idea of each paper; they do **not** reproduce the papers' numbers. **Every molecule, image and dataset is
> generated by code** — nothing is downloaded: the molecules are atomic models built from ideal α-helices with the
> size, mass and symmetry of the Protein Data Bank (PDB) entries they are named after (realistic stand-ins, not the
> real structures).

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Run** a Jupyter/Colab notebook, change a value, and re-run a cell (no prior coding needed).
2. **Place** molecules, viruses, cells, and microscope pixels on one scale from ångströms (Å) to micrometres (µm).
3. **Explain** the difference between *single-particle cryo-EM* and *cellular cryo-electron tomography (cryo-ET)*.
4. **Build** 3D density maps of a ribosome, a proteasome and RuBisCO, look at them, and estimate their size from
   their mass.
5. **Describe** the computational pipeline that turns a frozen cell into biological discoveries ("visual proteomics"),
   and name which Xu-lab papers address each step.
6. **Feel** why computation is necessary: try to spot molecules by eye in a simulated cellular tomogram.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | check the computer, import tools | 3 min |
| 1 · Notebook literacy | a 10-minute Python crash course | 10 min |
| 2 · Scales of life | put Å, nm and µm objects on one ruler | 10 min |
| 3 · Meet the molecular zoo | build density maps of six molecules and measure them | 15 min |
| 4 · Two ways to look | single-particle cryo-EM vs cryo-ET | 10 min |
| 5 · The pipeline and the research map | where each lab paper fits | 10 min |
| 6 · Teaser: find the ribosomes | a simulated crowded tomogram | 10 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 10 min |

## 🧫 The biological question

A bacterial cell is not a bag of dissolved proteins. It is a **crowded city**: about a fifth to a third of its interior
volume is occupied by large molecules — thousands of ribosomes making proteins, proteasomes and chaperones recycling and
folding them, enzymes such as RuBisCO fixing carbon — all jostling within a few hundred nanometres of each other.

Textbook structures come mostly from molecules **purified out of the cell**. But where are these machines *inside* a
living cell? Which ones are bound to a membrane, which ones form clusters, which shapes (conformations) do they adopt while
working, and how does a drug change all that? Answering these questions *in situ* (in place) is the goal of
**cryo-electron tomography (cryo-ET)** — and of a research programme called **visual proteomics**: identifying and
mapping every large molecule directly in 3D pictures of cells.

The microscope gives us the pictures. **Computation turns them into biology.** That second half is what the
Xu Lab works on, and what this course teaches.

---
## 0 · Environment setup

A notebook is a page of **text cells** (like this one) and **code cells** (grey boxes). To run a code cell, click it and
press **Shift + Enter** (or the ▶ button). Run the cells **in order, from the top**. The first cell checks what computer
you are on.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, this notebook only needs a CPU.")
try:
    import torch
    print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
except ImportError:
    print("PyTorch not installed (not needed in this notebook).")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("matplotlib", "matplotlib")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import os
import time

import matplotlib.pyplot as plt
import numpy as np
from scipy import ndimage

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "image.cmap": "gray"})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

Throughout the course we reuse a small set of helper functions: building molecules, rotating 3D volumes, adding
noise, simulating tomograms. They are collected in [`course_toolkit.py`](../course_toolkit.py). Each notebook **copies**
the helpers it needs into the next cell so that it runs on its own in Colab. You do **not** need to understand this cell
yet — every function in it is explained in a later module (the module number is in each docstring's first line where
relevant). Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import struct

import numpy as np
from scipy import ndimage

ZOO = {
    "4v4r": dict(name="70S ribosome", organism="bacterial (E. coli / T. thermophilus)", mass_kDa=2330, symmetry="C1"),
    "4v7r": dict(name="80S ribosome", organism="S. cerevisiae", mass_kDa=3330, symmetry="C1"),
    "5mrc": dict(name="mitochondrial ribosome", organism="S. cerevisiae", mass_kDa=3320, symmetry="C1"),
    "3j9i": dict(name="20S proteasome", organism="Thermoplasma acidophilum", mass_kDa=710, symmetry="D7"),
    "6utj": dict(name="singly capped proteasome", organism="Thermoplasma acidophilum", mass_kDa=910, symmetry="C7"),
    "4cr2": dict(name="26S proteasome", organism="S. cerevisiae", mass_kDa=1280, symmetry="C1"),
    "4v94": dict(name="TRiC/CCT chaperonin (×2)", organism="S. cerevisiae", mass_kDa=1910, symmetry="C2"),
    "1bxn": dict(name="RuBisCO (L4S4 half)", organism="Cupriavidus necator", mass_kDa=290, symmetry="C4"),
    "1qvr": dict(name="ClpB (3 of 6 subunits)", organism="Thermus thermophilus", mass_kDa=290, symmetry="C1"),
    "3cf3": dict(name="p97/VCP ATPase", organism="Mus musculus", mass_kDa=570, symmetry="C6"),
    "2cg9": dict(name="Hsp90–Sba1 chaperone", organism="S. cerevisiae", mass_kDa=190, symmetry="C2"),
    "1u6g": dict(name="Cand1–Cul1–Roc1 ligase", organism="H. sapiens", mass_kDa=260, symmetry="C1"),
    "3d2f": dict(name="Sse1p–Hsp70 chaperone", organism="S. cerevisiae / H. sapiens", mass_kDa=230, symmetry="C1"),
    "1s3x": dict(name="Hsp70 ATPase domain", organism="H. sapiens", mass_kDa=44, symmetry="C1"),
}
_BOND = dict(N_CA=1.458, CA_C=1.525, C_N=1.329, C_O=1.231)
_ANGLE = dict(N_CA_C=111.2, CA_C_N=116.2, C_N_CA=121.7, CA_C_O=120.5)
_SIDECHAIN = {
    "GLY": [], "ALA": [], "SER": [("OG", "O")], "CYS": [("SG", "S")], "VAL": [("CG1", "C"), ("CG2", "C")],
    "THR": [("OG1", "O"), ("CG2", "C")], "PRO": [("CG", "C"), ("CD", "C")],
    "LEU": [("CG", "C"), ("CD1", "C"), ("CD2", "C")], "ILE": [("CG1", "C"), ("CG2", "C"), ("CD1", "C")],
    "ASP": [("CG", "C"), ("OD1", "O"), ("OD2", "O")], "ASN": [("CG", "C"), ("OD1", "O"), ("ND2", "N")],
    "GLU": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("OE2", "O")], "GLN": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("NE2", "N")],
    "LYS": [("CG", "C"), ("CD", "C"), ("CE", "C"), ("NZ", "N")], "MET": [("CG", "C"), ("SD", "S"), ("CE", "C")],
    "ARG": [("CG", "C"), ("CD", "C"), ("NE", "N"), ("CZ", "C"), ("NH1", "N"), ("NH2", "N")],
    "HIS": [("CG", "C"), ("ND1", "N"), ("CD2", "C"), ("CE1", "C"), ("NE2", "N")],
    "PHE": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C")],
    "TYR": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C"), ("OH", "O")],
    "TRP": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("NE1", "N"), ("CE2", "C"), ("CE3", "C"), ("CZ2", "C"),
            ("CZ3", "C"), ("CH2", "C")],
}
_AA_FREQ = {"LEU": 9.9, "ALA": 8.3, "GLY": 7.1, "VAL": 6.9, "GLU": 6.8, "SER": 6.6, "ILE": 5.9, "LYS": 5.8, "ARG": 5.5,
            "ASP": 5.5, "THR": 5.3, "PRO": 4.7, "ASN": 4.1, "GLN": 3.9, "PHE": 3.9, "TYR": 2.9, "MET": 2.4, "HIS": 2.3,
            "CYS": 1.4, "TRP": 1.1}
def _nerf(a, b, c, bond, angle, torsion):
    """Place atom d so that |cd| = bond, angle(b,c,d) = angle and dihedral(a,b,c,d) = torsion (degrees)."""
    bc = (c - b) / np.linalg.norm(c - b)
    n = np.cross(b - a, bc)
    n /= np.linalg.norm(n)
    m = np.cross(n, bc)
    ang, tor = np.radians(angle), np.radians(torsion)
    d = np.array([-bond * np.cos(ang), bond * np.sin(ang) * np.cos(tor), bond * np.sin(ang) * np.sin(tor)])
    return c + d[0] * bc + d[1] * m + d[2] * n


def build_backbone(phi, psi, omega=None) -> np.ndarray:
    """Backbone atoms N, CA, C, O, CB (array n×5×3, Å) of a chain with the given torsion angles (degrees)."""
    n = len(phi)
    omega = np.full(n, 180.0) if omega is None else np.asarray(omega, float)
    N = [np.zeros(3)]
    CA = [np.array([_BOND["N_CA"], 0.0, 0.0])]
    C = [_nerf(np.array([0.0, 1.0, 0.0]), N[0], CA[0], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[0])]
    for i in range(n - 1):
        N.append(_nerf(N[i], CA[i], C[i], _BOND["C_N"], _ANGLE["CA_C_N"], psi[i]))
        CA.append(_nerf(CA[i], C[i], N[i + 1], _BOND["N_CA"], _ANGLE["C_N_CA"], omega[i]))
        C.append(_nerf(C[i], N[i + 1], CA[i + 1], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[i + 1]))
    N, CA, C = np.array(N), np.array(CA), np.array(C)
    O = np.array([_nerf(N[i], CA[i], C[i], _BOND["C_O"], _ANGLE["CA_C_O"], psi[i] + 180.0) for i in range(n)])
    b, c = CA - N, C - CA
    a = np.cross(b, c)
    CB = -0.58273431 * a + 0.56802827 * b - 0.54067466 * c + CA      # ideal Cβ position
    return np.stack([N, CA, C, O, CB], 1)


def _ideal_helix_template(n=160):
    """An ideal α-helix (φ = −57.8°, ψ = −47.0°) with its axis along +z through x = y = 0."""
    bb = build_backbone(np.full(n, -57.8), np.full(n, -47.0))
    ca = bb[:, 1]
    centre = ca.mean(0)
    u = np.linalg.svd(ca - centre)[2][0]
    if np.dot(ca[-1] - ca[0], u) < 0:
        u = -u
    # rotation taking u to z
    v = np.cross(u, [0, 0, 1.0]); s = np.linalg.norm(v); cth = u[2]
    K = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    Rm = np.eye(3) + K + K @ K * ((1 - cth) / (s ** 2 + 1e-12))
    xyz = (bb - centre) @ Rm.T
    axis_xy = xyz[:, 1, :2].mean(0)
    xyz[..., :2] -= axis_xy
    return xyz


_HELIX = _ideal_helix_template()
def _sidechains(bb, resnames, rng):
    """Approximate side-chain heavy atoms grown outward from Cβ (positions are schematic; counts/elements are right)."""
    names, elems, xyz, resid = [], [], [], []
    ca, cb = bb[:, 1], bb[:, 4]
    out = cb - ca
    out /= np.linalg.norm(out, axis=1, keepdims=True)
    for i, rn in enumerate(resnames):
        sc = _SIDECHAIN[rn]
        if not sc:
            continue
        perp = np.cross(out[i], rng.normal(size=3)); perp /= np.linalg.norm(perp)
        for k, (an, el) in enumerate(sc):
            depth = 1.0 + (k // 2)                               # schematic zig-zag outward
            pos = cb[i] + 1.5 * depth * out[i] + 0.9 * (1 if k % 2 else -1) * perp
            names.append(an); elems.append(el); xyz.append(pos); resid.append(i)
    return names, elems, np.array(xyz).reshape(-1, 3), np.array(resid, int)


def _loop_path(a, b, outward, step=3.8):
    """Cα positions of a loop from Cα `a` to Cα `b`: a circular arc bulging along `outward`, cut into equal chords so
    that every Cα–Cα distance (including the two ends) is exactly `step` Å."""
    c = np.linalg.norm(b - a)
    e = (b - a) / c
    u = outward - (outward @ e) * e                                  # bulge direction ⟂ to the chord
    if np.linalg.norm(u) < 1e-6:
        u = np.cross(e, [0.0, 0.0, 1.0]) if abs(e[2]) < 0.9 else np.cross(e, [1.0, 0.0, 0.0])
    u = u / np.linalg.norm(u)
    m = max(int(np.ceil(c / step)), 1)                               # m new residues → n = m + 1 equal chords
    n = m + 1
    lo, hi = 1e-9, 2 * np.pi - 1e-9                                  # arc angle Θ: sin(Θ/2) / sin(Θ/2n) = c / step
    for _ in range(60):
        th = (lo + hi) / 2
        lo, hi = (th, hi) if np.sin(th / 2) / np.sin(th / (2 * n)) > c / step else (lo, th)
    R = step / (2 * np.sin(th / (2 * n)))
    centre = (a + b) / 2 - R * np.cos(th / 2) * u
    beta = np.pi / 2 + th / 2 - th * np.arange(1, n) / n             # angles of the new points, a → b over the bulge
    return centre + R * (np.outer(np.cos(beta), e) + np.outer(np.sin(beta), u))


def _assemble_chain(ca_path_segments, rng, chain="A", start_resnum=1):
    """Turn helices (n×5×3 arrays) joined by loops into one chain with residue names and side chains."""
    blocks = []
    for k, seg in enumerate(ca_path_segments):
        blocks.append(seg)
        if k + 1 < len(ca_path_segments):
            a, b = seg[-1, 1], ca_path_segments[k + 1][0, 1]
            path = _loop_path(a, b, (a + b) / 2 - seg.reshape(-1, 3).mean(0))
            m = len(path)
            tang = np.gradient(path, axis=0) if m > 1 else np.array([b - a])
            tang /= np.linalg.norm(tang, axis=1, keepdims=True)
            side = np.cross(tang, rng.normal(size=3)); side /= np.linalg.norm(side, axis=1, keepdims=True)
            loop = np.stack([path - 1.2 * tang, path, path + 1.2 * tang, path + 1.2 * tang + 1.0 * side,
                             path - 1.3 * side], 1)
            blocks.append(loop)
    bb = np.concatenate(blocks, 0)
    aa = list(_AA_FREQ)
    p = np.array([_AA_FREQ[a] for a in aa]); p /= p.sum()
    resnames = rng.choice(aa, len(bb), p=p)
    bb_names = ["N", "CA", "C", "O", "CB"]
    names, elems, xyz, resid = [], [], [], []
    for i in range(len(bb)):
        for j, an in enumerate(bb_names):
            if an == "CB" and resnames[i] == "GLY":
                continue
            names.append(an); elems.append(an[0]); xyz.append(bb[i, j]); resid.append(i)
    sn, se, sx, sr = _sidechains(bb, resnames, rng)
    names += sn; elems += se; resid = np.concatenate([np.array(resid, int), sr])
    xyz = np.concatenate([np.array(xyz), sx], 0)
    order = np.argsort(resid, kind="stable")
    return dict(xyz=xyz[order].astype(np.float32), name=np.array(names)[order], element=np.array(elems)[order],
                resname=resnames[resid[order]], resnum=(resid[order] + start_resnum).astype(int),
                chain=np.full(len(order), chain))


def helix_bundle(radii_A, rng, chain="A", spacing=10.0):
    """A globular 'domain' that fills an ellipsoid (semi-axes in Å, long axis = local z) with antiparallel helices."""
    a, b, c = radii_A[0] + 2.5, radii_A[1] + 2.5, radii_A[2] + 1.5   # helices stop short of the surface: pad
    rows = []
    j = 0
    while (j * spacing * np.sqrt(3) / 2) <= b:
        rows.append(j); rows.append(-j) if j else None; j += 1
    pts = []
    for jj in sorted(set(rows)):
        y = jj * spacing * np.sqrt(3) / 2
        xs = np.arange(-a, a + 1e-6, spacing) + (spacing / 2 if jj % 2 else 0)
        row = [(x, y) for x in xs if (x / a) ** 2 + (y / b) ** 2 < 1]
        pts.append(row if (jj % 2 == 0) else row[::-1])
    pts = [p for row in pts for p in row]
    segs = []
    for k, (x, y) in enumerate(pts):
        h = c * np.sqrt(max(0.0, 1 - (x / a) ** 2 - (y / b) ** 2)) - 2.0
        n = int(2 * h / 1.5)
        if n < 6:
            continue
        seg = _HELIX[:n].copy()
        seg[..., 2] -= seg[:, 1, 2].mean()
        phase = rng.uniform(0, 2 * np.pi)
        Rz = np.array([[np.cos(phase), -np.sin(phase), 0], [np.sin(phase), np.cos(phase), 0], [0, 0, 1]])
        seg = seg @ Rz.T
        if len(segs) % 2 == 1:                                   # antiparallel neighbours: 180° turn about x
            seg = seg * np.array([1.0, -1.0, -1.0])
        seg = seg + np.array([x, y, rng.uniform(-1.0, 1.0)])
        segs.append(seg)
    return _assemble_chain(segs, rng, chain)


def _transform(atoms, R=np.eye(3), t=(0, 0, 0)):
    out = dict(atoms)
    out["xyz"] = (atoms["xyz"] @ np.asarray(R, float).T + np.asarray(t, float)).astype(np.float32)
    return out


def merge_atoms(parts):
    keys = parts[0].keys()
    return {k: np.concatenate([p[k] for p in parts]) for k in keys}


def _rz(a):
    return np.array([[np.cos(a), -np.sin(a), 0], [np.sin(a), np.cos(a), 0], [0, 0, 1.0]])


def _rx(a):
    return np.array([[1, 0, 0], [0, np.cos(a), -np.sin(a)], [0, np.sin(a), np.cos(a)]])


def _ry(a):
    return np.array([[np.cos(a), 0, np.sin(a)], [0, 1, 0], [-np.sin(a), 0, np.cos(a)]])


_CHAIN_IDS = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789"
def _ring(n, radius, z, radii, phase=0.0, tilt=None):
    """n identical subunits (semi-axes `radii` nm: radial, tangential, axial) around the z axis."""
    out = []
    for k in range(n):
        a = phase + 2 * np.pi * k / n
        R = _rz(a) if tilt is None else _rz(a) @ tilt
        out.append(dict(radii=radii, centre=(radius * np.cos(a), radius * np.sin(a), z), R=R, key=("ring", radii, tilt is None)))
    return out


def _lobe(radii, centre, R=None):
    return [dict(radii=radii, centre=centre, R=np.eye(3) if R is None else R, key=None)]


def _dihedral(subunits):
    """Add the partners generated by a 2-fold axis along x (dihedral symmetry)."""
    Rx = _rx(np.pi)
    return subunits + [dict(radii=s["radii"], centre=tuple(Rx @ np.array(s["centre"])), R=Rx @ s["R"], key=s["key"])
                       for s in subunits]


def _blueprint(pid):
    """Subunit layout (nm) of the code-generated stand-in for each zoo entry."""
    if pid in ("4v4r", "4v7r", "5mrc"):
        s = {"4v4r": 1.0, "4v7r": 1.13, "5mrc": 1.14}[pid]
        L = [_lobe((9.0 * s, 8.0 * s, 6.5 * s), (0, 0, 4.5 * s)),                       # large subunit body
             _lobe((3.0 * s, 3.0 * s, 3.0 * s), (0, 1.0 * s, 11.0 * s)),                # central protuberance
             _lobe((2.0 * s, 2.0 * s, 3.5 * s), (-9.0 * s, 0, 7.0 * s), _ry(-0.6)),     # L1-like stalk
             _lobe((1.4 * s, 1.4 * s, 4.0 * s), (9.5 * s, 0, 6.5 * s), _ry(0.7)),       # L7/L12-like stalk
             _lobe((7.5 * s, 5.0 * s, 3.8 * s), (1.0 * s, 0, -4.5 * s)),                # small subunit body
             _lobe((3.5 * s, 3.2 * s, 3.0 * s), (-5.5 * s, -1.0 * s, -6.5 * s))]        # small subunit head
        if pid == "4v7r":                                                                # eukaryotic expansion segments
            L += [_lobe((3.2, 2.6, 2.8), (8.0, 7.0, 7.0)), _lobe((3.0, 2.6, 3.2), (-6.0, -8.0, -6.5)),
                  _lobe((2.4, 2.4, 3.6), (-10.5, 4.0, -1.0), _ry(0.5))]
        if pid == "5mrc":                                                                # mitochondria-specific proteins
            L += [_lobe((4.2, 3.6, 2.8), (0, -7.5, 12.5)), _lobe((3.4, 3.0, 3.4), (9.0, -6.0, -8.0)),
                  _lobe((3.0, 3.0, 3.0), (-9.5, 7.0, 1.0)), _lobe((2.5, 4.5, 2.0), (2.0, 9.0, 9.0))]
        prng = np.random.default_rng(_seed(pid) + 1)                                     # rRNA-like surface bumps
        for _ in range(8):
            d = prng.normal(size=3); d /= np.linalg.norm(d)
            L += [_lobe(tuple(prng.uniform(1.2, 1.9, 3)), tuple(s * d * np.array([9.5, 8.0, 10.5])), _rz(prng.uniform(0, 6)))]
        return [x for l in L for x in l]
    if pid in ("3j9i", "6utj", "4cr2"):
        core = []
        for z, radii, ph in [(1.9, (1.9, 1.7, 2.0), 0.0), (5.6, (2.1, 1.7, 1.85), np.pi / 7)]:
            core += _ring(7, 3.6, z, radii, ph)
        core = _dihedral(core)                                                           # α7β7β7α7, D7
        if pid == "6utj":
            core += _ring(7, 3.3, 10.0, (1.9, 1.6, 2.4), np.pi / 14)                     # one heptameric cap: C7
        if pid == "4cr2":
            core += _ring(6, 4.3, 9.6, (2.2, 1.8, 2.0), 0.3)                             # base ring of six
            core += (_lobe((4.0, 3.0, 3.0), (3.5, 0, 14.5)) + _lobe((3.0, 2.5, 3.5), (-1.5, 3.0, 15.5))
                     + _lobe((2.5, 2.5, 2.5), (-3.5, -2.0, 13.5)) + _lobe((3.0, 3.0, 2.5), (-4.5, 0.5, 10.5)))
        return core
    if pid == "4v94":
        one = []
        for z in (3.6, -3.6):
            one += _ring(8, 5.2, z, (2.6, 2.0, 3.4), np.pi / 8 if z < 0 else 0.0)
        two = []
        for dx, rot in [(-8.5, 0.0), (8.5, np.pi)]:
            for s in one:
                c = _rz(rot) @ np.array(s["centre"])
                two.append(dict(radii=s["radii"], centre=(c[0] + dx, c[1], c[2]), R=_rz(rot) @ s["R"], key=s["key"]))
        return two
    if pid == "1bxn":
        # like the deposited file (the crystal's asymmetric unit): one L4S4 layer = half of the L8S8 enzyme, C4.
        # `_dihedral` of this list would give the complete enzyme (D4).
        return _ring(4, 3.6, 2.2, (2.6, 2.4, 2.4)) + _ring(4, 2.6, 5.0, (1.9, 1.7, 1.35), np.pi / 4)
    if pid == "1qvr":
        # like the deposited file: three neighbouring subunits (half of the C6 hexameric ring), C1
        tiers = [_ring(6, 4.2, 2.2, (2.2, 2.0, 2.0)), _ring(6, 3.9, -2.2, (2.3, 2.0, 2.0), np.pi / 6),
                 _ring(6, 7.2, 2.4, (0.9, 0.9, 4.6), 0.4, tilt=_rx(1.2)),               # coiled-coil 'propellers'
                 _ring(6, 3.8, 6.6, (1.7, 1.6, 1.8), 0.5)]                               # N-domains on top
        return [s for t in tiers for s in t[:3]]
    if pid == "3cf3":
        return (_ring(6, 4.0, 2.0, (2.1, 2.0, 2.0)) + _ring(6, 4.0, -2.2, (2.2, 2.0, 2.1), np.pi / 6)
                + _ring(6, 6.6, 3.0, (2.0, 1.6, 1.8), 0.25))                             # N-domains flush: C6
    if pid == "2cg9":
        half = (_lobe((1.9, 1.7, 1.7), (2.2, 0.8, 6.0)) + _lobe((2.2, 1.8, 2.0), (2.6, -0.4, 2.0))
                + _lobe((1.7, 1.5, 1.8), (1.4, 0.3, -2.2)) + _lobe((1.4, 1.2, 1.2), (4.2, 1.6, 7.6)))
        Rz = _rz(np.pi)
        return half + [dict(radii=h["radii"], centre=tuple(Rz @ np.array(h["centre"])), R=Rz @ h["R"], key=None) for h in half]
    if pid == "1u6g":
        out = []
        for k in range(6):                                                                # curved cullin stalk
            t = k / 5
            out += _lobe((1.4, 1.4, 1.6), (8.0 * np.sin(0.9 * t) - 3.0, 0, 9.0 * t - 4.5), _ry(0.9 * t))
        out += _lobe((2.2, 2.0, 2.0), (4.0, 0, 5.8)) + _lobe((1.2, 1.1, 1.1), (6.0, 1.8, 6.5))
        for k in range(10):                                                               # Cand1-like superhelix
            a = 1.6 * np.pi * k / 9
            out += _lobe((1.5, 1.4, 1.4), (-3.0 + 7.0 * np.sin(0.9 * k / 9) + 2.6 * np.cos(a), 2.6 * np.sin(a), 9.0 * k / 9 - 4.5))
        return out
    if pid == "3d2f":
        comp = (_lobe((1.8, 1.6, 1.8), (0, 0, 0)) + _lobe((1.8, 1.6, 1.8), (0, 3.2, 0)) + _lobe((1.6, 1.4, 2.4), (3.0, 1.6, 1.8))
                + _lobe((1.6, 1.5, 1.6), (-3.2, 1.0, -1.0)) + _lobe((1.6, 1.5, 1.6), (-3.2, 3.6, -0.6)))
        R2, t2 = _rx(2.3) @ _rz(0.7), np.array([1.5, -4.5, 5.5])
        return comp + [dict(radii=c["radii"], centre=tuple(R2 @ np.array(c["centre"]) + t2), R=R2 @ c["R"], key=None) for c in comp]
    if pid == "1s3x":
        return (_lobe((1.4, 1.4, 1.6), (-1.6, 0, -1.2)) + _lobe((1.3, 1.3, 1.5), (-1.4, 0.2, 1.6))
                + _lobe((1.4, 1.4, 1.6), (1.6, 0, -1.2)) + _lobe((1.2, 1.3, 1.4), (1.5, -0.3, 1.8)))
    raise KeyError(pid)


def _seed(pid):
    return int.from_bytes(pid.encode(), "little") % (2 ** 31)


_SCALE = {"4v4r": 0.923, "4v7r": 0.906, "5mrc": 0.883, "3j9i": 0.930, "6utj": 0.930, "4cr2": 0.930, "4v94": 0.886,
          "1bxn": 0.930, "1qvr": 0.885, "3cf3": 0.935, "2cg9": 0.995, "1u6g": 0.934, "3d2f": 0.997, "1s3x": 0.884}
_ATOMS_CACHE = {}
def zoo_atoms(pid):
    """Atomic model (dict of arrays) of the code-generated stand-in for zoo entry `pid` (Å, centred at the origin)."""
    pid = pid.lower()
    if pid in _ATOMS_CACHE:
        return _ATOMS_CACHE[pid]
    rng = np.random.default_rng(_seed(pid))
    built, parts = {}, []
    f = _SCALE.get(pid, 1.0)
    for k, s in enumerate(_blueprint(pid)):
        radii_A = tuple(10.0 * f * r for r in s["radii"])
        key = s["key"]
        if key is not None and key in built:
            sub = built[key]
        else:
            sub = helix_bundle(radii_A, rng)
            if key is not None:
                built[key] = sub
        part = _transform(sub, s["R"], 10.0 * f * np.array(s["centre"]))
        part["chain"] = np.full(len(part["xyz"]), _CHAIN_IDS[k % len(_CHAIN_IDS)])
        parts.append(part)
    atoms = merge_atoms(parts)
    if pid in ("3j9i", "6utj", "4cr2"):                             # lay the proteasome barrels along x (side view down z)
        atoms["xyz"] = (atoms["xyz"] @ _ry(np.pi / 2).T).astype(np.float32)
    atoms["xyz"] -= atoms["xyz"].mean(0)
    _ATOMS_CACHE[pid] = atoms
    return atoms


_Z = {"C": 6.0, "N": 7.0, "O": 8.0, "S": 16.0, "P": 15.0, "H": 1.0}
def atoms_to_map(xyz, weights=None, voxel=10.0, box=None, sigma_A=None):
    """Density map [z, y, x] from atom positions (Å): trilinear splatting on a grid twice as fine, Gaussian blur
    (σ = `sigma_A`, default = voxel/2), then 2×2×2 binning to `voxel` Å. Centred on the box if `box` is given."""
    xyz = np.asarray(xyz, float)
    w = np.ones(len(xyz)) if weights is None else np.asarray(weights, float)
    fine = voxel / 2.0
    if box is None:
        ext = np.ptp(xyz, 0).max() + 6 * voxel
        box = int(np.ceil(ext / voxel)) + 1
    n = 2 * box
    centre = (n - 1) / 2.0
    p = xyz[:, ::-1] / fine + centre                                  # (z, y, x) grid coordinates
    grid = np.zeros((n, n, n), np.float32)
    i0 = np.floor(p).astype(int)
    f = p - i0
    for dz in (0, 1):
        for dy in (0, 1):
            for dx in (0, 1):
                idx = i0 + np.array([dz, dy, dx])
                wt = w * np.where(dz, f[:, 0], 1 - f[:, 0]) * np.where(dy, f[:, 1], 1 - f[:, 1]) * np.where(dx, f[:, 2], 1 - f[:, 2])
                ok = np.all((idx >= 0) & (idx < n), 1)
                np.add.at(grid, tuple(idx[ok].T), wt[ok])
    grid = ndimage.gaussian_filter(grid, (sigma_A if sigma_A else voxel / 2.0) / fine)
    return grid.reshape(box, 2, box, 2, box, 2).mean((1, 3, 5)).astype(np.float32)


def zoo_map(pid, voxel="10A"):
    """(density[z, y, x], voxel size Å, source) of the code-generated stand-in for a zoo entry (no download)."""
    atoms = zoo_atoms(pid)
    v = float(str(voxel).rstrip("A"))
    w = np.array([_Z.get(e, 6.0) for e in atoms["element"]])
    vol = atoms_to_map(atoms["xyz"], w, voxel=v)
    return vol, v, f"code-generated stand-in modelled on PDB {pid.upper()}"


def prepare_map(vol: np.ndarray, size: int = 32, sigma: float = 1.0) -> np.ndarray:
    """Low-pass, centre (by centre of mass), crop/pad to ``size``³ and scale to [0, 1].

    Zoo maps are sharp sums of atoms; a Gaussian blur of ~1 voxel (1 nm)
    mimics the resolution of a cellular tomogram.
    """
    v = ndimage.gaussian_filter(np.clip(vol, 0, None).astype(np.float32), sigma)
    s = np.array(v.shape)
    src0 = np.maximum((s - size) // 2, 0)                  # where the crop starts in the input ...
    dst0 = np.maximum((size - s) // 2, 0)                  # ... and where the pad starts in the output
    n = np.minimum(s, size)
    com = np.array(ndimage.center_of_mass(v))
    shift = (size - 1) / 2 - dst0 + src0 - com             # puts the centre of mass exactly at the output's centre
    v = ndimage.shift(v, shift, order=1)
    out = np.zeros((size, size, size), np.float32)
    out[dst0[0]:dst0[0] + n[0], dst0[1]:dst0[1] + n[1], dst0[2]:dst0[2] + n[2]] = \
        v[src0[0]:src0[0] + n[0], src0[1]:src0[1] + n[1], src0[2]:src0[2] + n[2]]
    out -= out.min()
    return out / (out.max() + 1e-12)


def random_rotation(rng: np.random.Generator) -> np.ndarray:
    """Uniformly random proper rotation (det = +1) from a random unit quaternion."""
    q = rng.normal(size=4)
    q /= np.linalg.norm(q)
    w, x, y, z = q
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
        [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
        [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])


_P = np.array([[0, 0, 1], [0, 1, 0], [1, 0, 0]], float)  # (x,y,z) <-> (z,y,x)
def rotate_volume(vol: np.ndarray, R: np.ndarray, shift_xyz=(0, 0, 0), order: int = 1,
                  cval: float = 0.0) -> np.ndarray:
    """Rotate a volume about its centre by R (then translate by shift, in voxels, (x,y,z)).

    out(p) = vol(Rᵀ (p − c − t) + c): the *object* is rotated by R and moved by t.
    """
    R_idx = _P @ R @ _P                       # same rotation written in (z, y, x) index order
    c = (np.array(vol.shape) - 1) / 2.0
    t_idx = _P @ np.asarray(shift_xyz, float)
    M = R_idx.T                               # inverse rotation maps output -> input
    offset = c - M @ (c + t_idx)
    return ndimage.affine_transform(vol, M, offset=offset, order=order, mode="constant", cval=cval)


def freq_grid(shape):
    """Frequency coordinates (cycles/voxel) for an unshifted FFT; returns list per axis."""
    return np.meshgrid(*[np.fft.fftfreq(n) for n in shape], indexing="ij")


def missing_wedge_mask(shape, tilt_max_deg: float = 60.0) -> np.ndarray:
    """Binary Fourier mask of a single-axis tilt series about y (±tilt_max).

    A coefficient (k_z, k_y, k_x) is measured if |k_z| ≤ |k_x|·tan(tilt_max)
    — the formula M(ξ) = 1(|ξ_3| ≤ |ξ_1| tan θ) used by Xu et al. (2012).
    For 2D images [z, x] pass a 2-tuple shape.
    """
    k = freq_grid(shape)
    kz, kx = k[0], k[-1]
    m = np.abs(kz) <= np.abs(kx) * np.tan(np.radians(tilt_max_deg)) + 1e-9
    m[(0,) * len(shape)] = True               # keep the mean (DC) term
    return m.astype(np.float32)


def apply_fourier_mask(vol: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Multiply the FFT of ``vol`` by ``mask`` (unshifted layout) and return the real part."""
    return np.real(np.fft.ifftn(np.fft.fftn(vol) * mask)).astype(np.float32)


def add_noise_snr(signal: np.ndarray, snr: float, rng: np.random.Generator) -> np.ndarray:
    """Add white Gaussian noise so that var(signal)/var(noise) = snr."""
    sigma = np.sqrt(signal.var() / snr)
    return (signal + rng.normal(0, sigma, signal.shape)).astype(np.float32)


def electron_wavelength_A(kV: float = 300.0) -> float:
    """Relativistic electron wavelength in Å (0.0197 Å at 300 kV)."""
    V = kV * 1e3
    return 12.2643 / np.sqrt(V * (1 + 0.978466e-6 * V))


def ctf_2d(shape, pixel_A: float, defocus_um: float = 3.0, kV: float = 300.0,
           Cs_mm: float = 2.7, amp_contrast: float = 0.07, B: float = 0.0) -> np.ndarray:
    """Contrast transfer function on an unshifted FFT grid (underfocus is positive).

    CTF(k) = −[√(1−A²)·sin χ(k) + A·cos χ(k)]·exp(−B k²/4),
    χ(k) = π λ Δf k² − (π/2) C_s λ³ k⁴, with k in 1/Å.
    """
    lam = electron_wavelength_A(kV)
    k2 = sum(kk**2 for kk in freq_grid(shape)) / pixel_A**2
    dz = defocus_um * 1e4
    cs = Cs_mm * 1e7
    chi = np.pi * lam * dz * k2 - 0.5 * np.pi * cs * lam**3 * k2**2
    ctf = -(np.sqrt(1 - amp_contrast**2) * np.sin(chi) + amp_contrast * np.cos(chi))
    return (ctf * np.exp(-B * k2 / 4)).astype(np.float32)


def support_radius(t: np.ndarray, level: float = 0.1) -> float:
    """Radius (voxels) of the smallest centred sphere containing all voxels above ``level``·max."""
    pts = np.argwhere(t > level * t.max())
    if len(pts) == 0:
        return t.shape[0] / 2
    c = (np.array(t.shape) - 1) / 2
    return float(np.linalg.norm(pts - c, axis=1).max()) + 1.0


def make_crowded_cell(templates: dict, counts: dict, shape=(48, 160, 160), rng=None,
                      membrane: bool = True, tries_per_particle: int = 400):
    """Pack randomly rotated templates into a volume without overlap (bounding spheres).

    A simple rejection-sampling stand-in for the packing used by Pei et al. (2016)
    and Liu et al. (2020).  Particles are placed in random class order so that every
    class gets a fair chance.  Returns (clean_volume, particles) where each particle
    is dict(cls, center_zyx, R).  Optionally adds a spherical membrane vesicle.
    """
    rng = rng or np.random.default_rng(0)
    vol = np.zeros(shape, np.float32)
    occ_c, occ_r, particles = [], [], []
    if membrane:  # a vesicle: thin shell, ~4 nm thick lipid bilayer at 1 nm/voxel
        zc, yc, xc = shape[0] / 2, shape[1] * 0.78, shape[2] * 0.22
        rad = min(shape[1], shape[2]) * 0.16
        g = np.meshgrid(*[np.arange(n) for n in shape], indexing="ij")
        d = np.sqrt((g[0] - zc)**2 + (g[1] - yc)**2 + (g[2] - xc)**2)
        vol += 0.6 * np.exp(-((d - rad)**2) / (2 * 1.5**2))
        occ_c.append(np.array([zc, yc, xc])); occ_r.append(rad + 3)
    radii = {cls: support_radius(templates[cls], level=0.35) for cls in counts}
    order = [cls for cls, n in counts.items() for _ in range(n)]
    rng.shuffle(order)
    for cls in order:
        t, rad_t = templates[cls], radii[cls]
        for _ in range(tries_per_particle):
            c = np.array([rng.uniform(rad_t, s - rad_t) if s > 2 * rad_t else s / 2 for s in shape])
            if any(np.linalg.norm(c - oc) < rad_t + orr for oc, orr in zip(occ_c, occ_r)):
                continue
            R = random_rotation(rng)
            sub = rotate_volume(t, R)
            h = np.array(sub.shape) // 2
            lo = np.round(c).astype(int) - h
            sl_v, sl_s = [], []
            for a in range(3):
                a0, a1 = max(lo[a], 0), min(lo[a] + sub.shape[a], shape[a])
                sl_v.append(slice(a0, a1)); sl_s.append(slice(a0 - lo[a], a1 - lo[a]))
            vol[tuple(sl_v)] += sub[tuple(sl_s)]
            occ_c.append(c); occ_r.append(rad_t)
            particles.append(dict(cls=cls, center_zyx=c, R=R))
            break
    return vol, particles


def simulate_tomogram(clean: np.ndarray, rng, snr: float = 0.05, tilt_max_deg: float = 60.0,
                      ctf_pixel_A: float | None = 10.0, defocus_um: float = 5.0) -> np.ndarray:
    """Fast tomogram simulation: (optional 3D CTF-like filter) → missing wedge → noise.

    The CTF is applied isotropically here (a simplification of per-tilt CTF).
    """
    v = clean
    if ctf_pixel_A:
        ctf = ctf_2d(clean.shape, ctf_pixel_A, defocus_um=defocus_um, kV=300, B=0)
        v = apply_fourier_mask(v, -ctf)          # sign flip: dense objects appear bright
    v = apply_fourier_mask(v, missing_wedge_mask(v.shape, tilt_max_deg))
    return add_noise_snr(v, snr, rng)
print("🧰 toolkit loaded")

---
## 1 · Notebook literacy: a 10-minute Python crash course

You will never need to memorise syntax in this course — you will mostly *read* code, run it, and change numbers. Five
ideas cover almost everything.

| Idea | Python | Biology analogy |
|---|---|---|
| **Variable** — a named value | `pixel_size_A = 10.0` | a labelled tube |
| **List** — an ordered collection | `masses = [2330, 710, 560]` | a rack of tubes |
| **Loop** — repeat for each item | `for m in masses: ...` | pipetting into every well |
| **Function** — a reusable recipe | `def volume_from_mass(m): ...` | a lab protocol |
| **Array** — a grid of numbers (NumPy) | `np.zeros((32, 32, 32))` | a 3D image |

A line starting with `#` is a **comment**: notes for humans, ignored by Python.

In [ ]:
# ── 1.1 Variables, arithmetic and printing ──
ribosome_mass_kDa = 2330            # mass of the bacterial 70S ribosome (PDB 4V4R), kilodaltons
proteasome_mass_kDa = 710           # 20S proteasome (PDB 3J9I)
ratio = ribosome_mass_kDa / proteasome_mass_kDa
print("The ribosome is", round(ratio, 1), "times heavier than the 20S proteasome.")
print(f"Same thing with an f-string: {ratio:.1f}×")   # f-strings put values inside text

In [ ]:
# ── 1.2 Lists, loops and functions ──
def volume_from_mass(mass_kDa: float) -> float:
    """Volume (nm³) of a protein of the given mass, assuming density 1.35 g/cm³ (≈ 0.81 Da/Å³)."""
    daltons = mass_kDa * 1000
    volume_A3 = daltons / 0.81          # 1 Å³ holds about 0.81 Da of protein
    return volume_A3 / 1000             # 1 nm³ = 1000 Å³


names = ["70S ribosome", "20S proteasome", "RuBisCO", "Hsp70 ATPase domain"]
masses = [2330, 710, 560, 43]
for name, m in zip(names, masses):              # zip walks through two lists in step
    v = volume_from_mass(m)
    diameter = 2 * (3 * v / (4 * np.pi)) ** (1 / 3)   # diameter of a sphere with that volume
    print(f"{name:22s} {m:6d} kDa → {v:7.0f} nm³ → sphere of diameter ≈ {diameter:4.1f} nm")

🔍 **What just happened?** A *function* (`volume_from_mass`) packaged a protocol — "convert mass to volume" — that we
applied in a *loop* to every molecule. Proteins have nearly the same density (≈ 1.35 g/cm³), so **mass tells you size**.
A ribosome is ~18 nm across *if it were a sphere*; real ribosomes are ~20–25 nm because they are not spheres (and RNA is
denser than protein, so this is only a rough estimate).

✏️ **Your turn.** Add a molecule to the two lists (for example the 26S proteasome, ≈ 2,500 kDa when fully assembled,
or a virus capsid) and re-run the cell. What happens if the two lists have different lengths? *(Try it — reading error
messages is a skill.)*

In [ ]:
# ── 1.3 Arrays: images are grids of numbers ──
tiny_image = np.zeros((5, 7))          # 5 rows (y) × 7 columns (x), all zeros
tiny_image[2, 3] = 1.0                 # set the pixel at row 2, column 3 (counting from 0!)
print(tiny_image)
print("shape:", tiny_image.shape, "| total:", tiny_image.sum())

fig, ax = plt.subplots(figsize=(3, 2.2))
ax.imshow(tiny_image)
ax.set_title("a 5×7 'image' with one bright pixel")
ax.set_xlabel("x (column)"); ax.set_ylabel("y (row)")
plt.show()

💡 Every image in this course is an **array**: a 2D grid for a micrograph `img[y, x]`, a 3D grid for a tomogram
`vol[z, y, x]`. Counting starts at **0**. Module 01 goes deeper.

---
## 2 · Scales of life: from ångströms to micrometres

Structural biology spans five orders of magnitude. The units:

- **1 Å** (ångström) = 0.1 nm = 10⁻¹⁰ m — about the size of an atom; bond lengths are 1–1.5 Å.
- **1 nm** (nanometre) = 10 Å — a small protein is a few nm.
- **1 µm** (micrometre) = 1,000 nm — a bacterium is 1–2 µm long.

Because these numbers differ so much, we plot them on a **logarithmic axis**, where each step is ×10.

In [ ]:
# ── 2.1 One ruler for molecules, cells and pixels ──
objects = [  # (label, size in nm, category)
    ("C–C bond (1.5 Å)", 0.15, "atomic"),
    ("water molecule", 0.28, "atomic"),
    ("amino-acid step along a chain (3.8 Å)", 0.38, "atomic"),
    ("α-helix width", 1.0, "atomic"),
    ("Hsp70 ATPase domain (43 kDa)", 4.0, "molecule"),
    ("RuBisCO (L₈S₈, ≈ 560 kDa)", 11.0, "molecule"),
    ("20S proteasome (length)", 15.0, "molecule"),
    ("70S ribosome", 21.0, "molecule"),
    ("lipid bilayer thickness", 4.0, "molecule"),
    ("SARS-CoV-2 virion", 100.0, "cell"),
    ("FIB-milled lamella thickness", 200.0, "cell"),
    ("E. coli cell length", 2000.0, "cell"),
    ("raw detector pixel (≈1.3 Å)", 0.13, "microscope"),
    ("binned tomogram voxel (≈1 nm)", 1.0, "microscope"),
    ("typical cellular tomogram resolution", 4.0, "microscope"),
]
colors = {"atomic": "tab:blue", "molecule": "tab:green", "cell": "tab:red", "microscope": "tab:purple"}
fig, ax = plt.subplots(figsize=(7.5, 4.6))
for i, (lab, size, cat) in enumerate(sorted(objects, key=lambda o: o[1])):
    ax.scatter(size, i, color=colors[cat], s=30)
    ax.text(size * 1.25, i, lab, va="center", fontsize=7)
ax.set_xscale("log"); ax.set_xlim(0.08, 2e5); ax.set_yticks([])
ax.set_xlabel("size (nm, log scale: each tick ×10)")
for cat, col in colors.items():
    ax.scatter([], [], color=col, label=cat)
ax.legend(loc="lower right", fontsize=7)
ax.set_title("From atoms to cells: the sizes cryo-EM/ET must bridge")
plt.show()

🔍 **What just happened?** The microscope pixels (purple) sit right next to the molecules (green). A cellular tomogram
with ~1 nm voxels resolves the *shape* of a ribosome (≈ 20 voxels across) but not its atoms; to see atoms we must
**average many copies** (Module 14) — the central trick of the whole field.

---
## 3 · Meet the molecular zoo

A **density map** is a 3D array whose values say "how much matter (electrostatic potential) is here". Maps are computed
from atomic models (PDB entries) or measured by cryo-EM. We will use the same cast of molecules throughout the course:

| PDB | molecule | organism | why it matters |
|---|---|---|---|
| 4V4R | 70S ribosome | bacterial | makes all proteins; the most studied particle in cellular cryo-ET |
| 3J9I | 20S proteasome | *Thermoplasma acidophilum* | barrel-shaped protein shredder; 7-fold symmetric |
| 6UTJ | singly capped proteasome | *T. acidophilum* | the same barrel with a regulatory cap: a *compositional* variant |
| 1BXN | RuBisCO | *Cupriavidus necator* | fixes CO₂; one of the most abundant enzymes on Earth (the file — and our stand-in — holds half of it, Module 01) |
| 1QVR | ClpB | *Thermus thermophilus* | a disaggregase chaperone (the file holds 3 of its 6 subunits) — our "unknown" guest in later modules |
| 1S3X | Hsp70 ATPase domain | human | small (43 kDa): a lesson in what is *too small to see* |

Nothing is downloaded: for each entry the toolkit **builds an atomic model by code** — chains of amino acids folded
into ideal α-helices, packed into subunits of the right size and arranged with the right symmetry (7-fold rings for the
proteasome, a 4-fold double layer for RuBisCO, …) — and converts the atoms into a density map with 10 Å (1 nm) voxels
(Module 01 shows how). These stand-ins have the real complexes' mass, size and symmetry but not their exact shape.

In [ ]:
# ── 3.1 Build the zoo ──
CAST = ["4v4r", "3j9i", "6utj", "1bxn", "1qvr", "1s3x"]   # the molecules we use in this notebook
t0 = time.time()
raw_maps = {}
for pid in CAST:
    vol, voxel_A, source = zoo_map(pid)
    raw_maps[pid] = vol
    print(f"{pid.upper()}  {ZOO[pid]['name']:26s} array {str(vol.shape):14s} voxel {voxel_A:.0f} Å  ← {source}")
print(f"done in {time.time() - t0:.1f} s")

In [ ]:
# ── 3.2 Look at each molecule from three directions ──
zoo = {pid: prepare_map(v, size=40, sigma=1.0) for pid, v in raw_maps.items()}
fig, axes = plt.subplots(3, len(zoo), figsize=(1.6 * len(zoo), 5))
for j, (pid, v) in enumerate(zoo.items()):
    for i, (axis, lab) in enumerate([(0, "view down z"), (1, "view down y"), (2, "view down x")]):
        axes[i, j].imshow(v.sum(axis=axis))          # a projection = sum along one axis
        axes[i, j].set_xticks([]); axes[i, j].set_yticks([])
        if j == 0:
            axes[i, j].set_ylabel(lab, fontsize=8)
    axes[0, j].set_title(f"{pid.upper()}\n{ZOO[pid]['name']}", fontsize=8)
fig.suptitle("Projections of the molecular zoo (40 × 40 nm boxes, 1 nm voxels)", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Each panel is a **projection**: the sum of density along one direction — exactly what an
electron microscope records (Module 06). Notice how the same molecule looks *different* from different directions: the
20S proteasome is a rectangle from the side and a ring from the top. In a cell every copy points in a random direction,
which is why **alignment** (Modules 12, 13, 21) is so central.

### 3.3 How big is each molecule? Contouring by mass

When biologists display a density map as a surface they must choose a **threshold** (contour level). A principled
choice: pick the threshold at which the enclosed volume equals the volume expected from the molecule's mass,

$$V_{\text{expected}} \;=\; \frac{M\ [\text{Da}]}{0.81\ \text{Da}/\text{Å}^3}.$$

*In words:* protein packs about 0.81 dalton into every cubic ångström, so a known mass predicts a volume; we then lower
the threshold until that many voxels are "inside".

In [ ]:
# ── 3.3 Find the mass-based contour level for each molecule ──
def mass_threshold(density: np.ndarray, mass_kDa: float, voxel_A: float = 10.0) -> tuple[float, int]:
    """Threshold at which the number of voxels above it equals the volume expected from the mass."""
    n_expected = int(round(mass_kDa * 1000 / 0.81 / voxel_A**3))
    values = np.sort(density.ravel())[::-1]          # brightest voxels first
    return float(values[n_expected - 1]), n_expected


rows = []
for pid, v in zoo.items():
    thr, n_vox = mass_threshold(v, ZOO[pid]["mass_kDa"])
    inside = v >= thr
    extent_nm = np.ptp(np.argwhere(inside), axis=0) + 1            # bounding-box size in voxels = nm
    rows.append((pid, ZOO[pid]["mass_kDa"], n_vox, thr, extent_nm))
    print(f"{pid.upper()} {ZOO[pid]['name']:26s} {ZOO[pid]['mass_kDa']:5d} kDa → {n_vox:5d} voxels inside; "
          f"bounding box ≈ {extent_nm[2]}×{extent_nm[1]}×{extent_nm[0]} nm (x×y×z)")

fig, axes = plt.subplots(1, len(zoo), figsize=(1.6 * len(zoo), 1.9))
for ax, (pid, _, _, thr, _) in zip(axes, rows):
    ax.imshow((zoo[pid] >= thr).sum(axis=1) > 0)                   # silhouette of the contoured molecule
    ax.set_title(pid.upper(), fontsize=8); ax.axis("off")
fig.suptitle("Silhouettes at the mass-based contour level", fontsize=9)
plt.tight_layout(); plt.show()

🔍 **What just happened?** With one physical assumption (protein density) we turned a mass into a 3D shape and measured
the molecules in nanometres. The ribosome's bounding box is ≈ 16–23 nm, the Hsp70 domain only a few nm — at 1 nm voxels
and realistic noise, molecules below ~200–300 kDa are very hard to see in cellular tomograms.

✏️ **Your turn.** Change `sigma=1.0` to `sigma=2.5` in cell 3.2 (more blur, i.e. worse resolution) and re-run 3.2 and 3.3.
Do the silhouettes change? Which molecule loses its characteristic shape first?

---
## 4 · Two ways to look: single-particle cryo-EM vs cellular cryo-ET

Both techniques freeze the sample so fast (≈ 10⁵–10⁶ K/s) that water becomes **vitreous ice** — a glass, without
crystals — preserving molecules in a near-native state. Both use very few electrons because electrons destroy
biological molecules (**radiation damage**), so every image is extremely noisy. They differ in *what* is imaged:

| | **Single-particle cryo-EM** | **Cellular cryo-electron tomography (cryo-ET)** |
|---|---|---|
| Sample | purified molecules, many identical copies in a thin ice layer | a whole small cell, or a thin slice (lamella) of a larger cell |
| Images | one 2D projection per copy, many copies in random orientations | one object imaged from many **tilt angles** (e.g. −60°…+60°) |
| 3D comes from | combining projections of *different* copies | combining tilts of the *same* region |
| Typical result | near-atomic maps (2–4 Å) | a 3D "tomogram" of the cell (~2–5 nm), then **subtomogram averaging** of repeated molecules (≈ 3–30 Å) |
| Strength | resolution | **context**: where molecules are, what they touch, in which states |
| Main computational problems | particle picking, CTF, orientation search, heterogeneity | missing wedge, crowding, extreme noise, identifying unknown molecules |

The next cell draws both experiments schematically.

In [ ]:
# ── 4.1 A schematic of the two experiments ──
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.4))
# single particle: many copies, beam from the top
proj = zoo["3j9i"]
for k in range(9):
    x0, y0 = (k % 3) * 1.2, (k // 3) * 1.0
    ang = rng.uniform(0, 180)
    img = ndimage.rotate(proj.sum(axis=int(rng.integers(0, 3))), ang, reshape=False)
    a1.imshow(img, extent=(x0, x0 + 1, y0, y0 + 0.9), cmap="gray")
a1.annotate("", xy=(1.8, 3.15), xytext=(1.8, 3.6), arrowprops=dict(arrowstyle="->", lw=2, color="tab:orange"))
a1.text(1.95, 3.35, "electron beam", color="tab:orange", fontsize=8)
a1.set_xlim(-0.1, 3.6); a1.set_ylim(-0.1, 3.7); a1.axis("off")
a1.set_title("Single-particle cryo-EM:\nmany identical copies, random orientations", fontsize=9)
# tomography: one object, many tilts
for ang in [-60, -30, 0, 30, 60]:
    t = np.radians(ang)
    a2.plot([0, 2.2 * np.sin(t)], [0, 2.2 * np.cos(t)], color="tab:orange", lw=1)
    a2.text(2.4 * np.sin(t), 2.4 * np.cos(t), f"{ang}°", fontsize=7, ha="center")
a2.add_patch(plt.Rectangle((-0.9, -0.12), 1.8, 0.24, color="lightsteelblue"))
a2.text(0, -0.45, "the same cell / lamella, tilted", ha="center", fontsize=8)
a2.fill_between([-0.02, 0.02], [0, 0], [0, 0])
a2.set_xlim(-2.6, 2.6); a2.set_ylim(-0.7, 2.7); a2.set_aspect("equal"); a2.axis("off")
a2.set_title("Cryo-electron tomography:\none unique specimen, many tilt angles", fontsize=9)
plt.tight_layout(); plt.show()

💡 Tilting stops at about ±60–70° because the specimen holder and the growing thickness of the tilted slab block the
beam. The directions never measured form the **missing wedge** — you will meet it in almost every paper (Module 07).

---
## 5 · The computational pipeline and the research map

From a frozen cell to a discovery, data flow through a chain of computational steps. The table lists each step, what
goes wrong without computation, the Xu-lab papers that address it, and the module of this course that teaches it.

| # | Step | Why it is hard | Representative Xu-lab work | Module |
|---|---|---|---|---|
| 1 | **Simulate** realistic tomograms (ground truth for testing) | real cells have no answer key | crowded-cell simulation (Pei 2016; Liu 2020), SHREC benchmarks, CryoETGAN, DUAL | 08, 18, 22 |
| 2 | **Reconstruct** a tomogram from tilts | missing wedge, misaligned tilts | marker-free tilt alignment (Han 2019) | 07 |
| 3 | **Denoise** | SNR ≈ 0.01–0.1, no clean data | J-invariant volume shuffle (2025), DUAL (2024), score matching (2026) | 22 |
| 4 | **Pick** particles | crowding, unknown shapes, few labels | DoG & template matching, TomoPicker (2025), CryoMAE (2025), RobPicker | 10, 23 |
| 5 | **Align** subtomograms | 6D search, missing wedge | fast constrained matching (2012), gradient refinement (2012), Gum-Net (2020), BOE-ViT (2025) | 11, 12, 19 |
| 6 | **Average & classify** | mixtures of molecules and states | ML integration FAML (2018), deep classifiers (2017–2024), Harmony (2022) | 13, 15, 21 |
| 7 | **Discover** without templates | unknown complexes | template-free detection (2011), de novo pattern mining (2019), DISCA (2023) | 14, 20 |
| 8 | **Segment** membranes & organelles | thin, noisy, topologically complex | CryoSAM (2024), Vox-UDA (2025), Stable-Diffusion segmentation (2026), TopoFuse (2026) | 16, 18, 24 |
| 9 | **Validate** | noise can masquerade as signal | FSC & local resolution (2020), particle selection (2020) | 09 |
| 10 | **Interpret** biologically | from maps to mechanisms | fungal plasma membrane & antifungal action (2025) | 25 |

All of these tools are gathered in the lab's open-source toolkit **AITom** (<https://github.com/xulabs/aitom>), which grew
out of the earlier **TomoMiner** platform. The capstone (Module 34) chains the steps together.

### 📄 Paper connection — the big picture papers

**Beck, Topf, Frazier, Tjong, Xu, Zhang & Alber (2011).** *Exploring the spatial and temporal organization of a cell's
proteome.* J. Struct. Biol. 173(3):483–496. [doi:10.1016/j.jsb.2010.11.011](https://doi.org/10.1016/j.jsb.2010.11.011)
- **What it is:** a *review* (not a methods paper) of how cryo-ET, template matching and computational modelling could
  map which molecules sit where in a cell — the "visual proteomics" agenda this course is built around.
- **Why read it first:** it frames the questions (abundance, location, interactions, conformations) that later methods answer.

**Kim, Uddin, Xu & Chang (2023).** *Computational methods toward unbiased pattern mining and structure determination in
cryo-electron tomography data.* J. Mol. Biol. 435(9):168068. [doi:10.1016/j.jmb.2023.168068](https://doi.org/10.1016/j.jmb.2023.168068)
- **What it is:** a review organised along the pipeline above (segmentation → picking → structure determination →
  classification), contrasting template-based tools with *unbiased* (template-free) ones such as DISCA, Jim-Net and
  Harmony, and listing open challenges (few annotations, simulation, domain adaptation, accessibility, benchmarks).
- **Use it as:** your map of the field; re-read it after Module 15 and after Module 32.

**AITom.** Zeng & Xu (2019), *AITom: Open-source AI platform for cryo-electron tomography data analysis*,
[arXiv:1911.03044](https://arxiv.org/abs/1911.03044); Zhan, Zeng, Uddin & Xu (2025), *AITom: AI-guided cryo-electron
tomography image analyses toolkit*, J. Struct. Biol. 217(2):108207, [doi:10.1016/j.jsb.2025.108207](https://doi.org/10.1016/j.jsb.2025.108207).
- **What it is:** the lab's software platform with modules for simulation, picking, alignment, averaging, classification
  (supervised, few-label and unsupervised), segmentation and model interpretation; derived from the TomoMiner library.

**Book chapters** for a textbook-style overview: Han & Xu (2024) *The advance of computational methods in cryo-electron
tomography*, in *Frontiers in Bioimage Informatics Methodology*, pp. 3–54; Uddin, Dip, Jha, Zeng & Xu (2025) *Feature
detection in cryo-electron tomography image analysis*, in *Cryo-electron Tomography* (Academic Press), pp. 173–215.

---
## 6 · Teaser: can you find the ribosomes?

Let us build a tiny **in silico cell**: a 160 × 160 × 64 nm box packed with ribosomes, proteasomes and RuBisCO in random
orientations, plus a membrane vesicle. Then we "image" it the way a cryo-electron tomogram would look: we remove the
missing wedge, apply the microscope's contrast transfer function, and add noise at a realistic signal-to-noise ratio.
(Modules 06–09 explain each step; here we only look.)

In [ ]:
# ── 6.1 Build and image an in silico cell ──
t0 = time.time()
cast = {pid: prepare_map(raw_maps[pid], size=32, sigma=1.0) for pid in ["4v4r", "3j9i", "6utj", "1bxn"]}
counts = {"4v4r": 10, "3j9i": 8, "6utj": 6, "1bxn": 14}
clean, particles = make_crowded_cell(cast, counts, shape=(64, 160, 160), rng=np.random.default_rng(SEED))
tomogram = simulate_tomogram(clean, np.random.default_rng(SEED + 1), snr=0.1, tilt_max_deg=60)
print(f"placed {len(particles)} particles in {time.time() - t0:.1f} s; "
      f"occupancy ≈ {100 * (clean > 0.1).mean():.0f} % of voxels")
for pid in counts:
    print(f"  {ZOO[pid]['name']:26s}: {sum(p['cls'] == pid for p in particles)}")

In [ ]:
# ── 6.2 Look at a slice: first without the answer key ──
z_mid = clean.shape[0] // 2
fig, axes = plt.subplots(1, 2, figsize=(9, 4.4))
axes[0].imshow(clean[z_mid]); axes[0].set_title("the true density (never available in reality)")
axes[1].imshow(tomogram[z_mid]); axes[1].set_title("what a tomogram slice looks like (SNR 0.1)")
for ax in axes:
    ax.set_xlabel("x (nm)"); ax.set_ylabel("y (nm)")
plt.tight_layout(); plt.show()

🤔 **Try before you scroll.** In the right panel, how many ribosomes (the largest blobs) can you find? Write down your
count, then run the next cell for the answer key.

In [ ]:
# ── 6.3 Reveal the answer key ──
marker = {"4v4r": ("o", "tab:red"), "3j9i": ("s", "tab:blue"), "6utj": ("D", "tab:cyan"), "1bxn": ("^", "tab:green")}
fig, ax = plt.subplots(figsize=(5.2, 5))
ax.imshow(tomogram[z_mid])
for p in particles:
    z, y, x = p["center_zyx"]
    if abs(z - z_mid) < 9:                              # particles that intersect this slice
        m, c = marker[p["cls"]]
        ax.scatter(x, y, marker=m, facecolors="none", edgecolors=c, s=160, lw=1.5)
for pid, (m, c) in marker.items():
    ax.scatter([], [], marker=m, facecolors="none", edgecolors=c, label=ZOO[pid]["name"])
ax.legend(fontsize=7, loc="upper right"); ax.set_title("ground truth for this slice")
ax.set_xlabel("x (nm)"); ax.set_ylabel("y (nm)")
plt.show()

In [ ]:
# ── 6.4 The side view: the missing wedge smears objects along z ──
y_cut = int(np.median([p["center_zyx"][1] for p in particles if p["cls"] == "4v4r"]))
fig, axes = plt.subplots(2, 1, figsize=(8, 3.6))
axes[0].imshow(clean[:, y_cut, :], aspect="equal"); axes[0].set_title(f"true density, xz slice at y = {y_cut} nm")
axes[1].imshow(tomogram[:, y_cut, :], aspect="equal"); axes[1].set_title("simulated tomogram, same slice")
for ax in axes:
    ax.set_ylabel("z (nm)")
axes[1].set_xlabel("x (nm)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Even in a *simulated* cell with only four kinds of molecules — and a generous SNR of 0.1 —
finding and recognising molecules by eye is hard, and in the side view (xz) everything is smeared vertically by the
missing wedge. Real cells contain hundreds of kinds of molecules, real noise is worse, and a single dataset can hold
hundreds of thousands of particles. This is why **every step of the pipeline in Section 5 is a research problem**.

✏️ **Your turn.** In cell 6.1 change `snr=0.1` to `snr=0.02` (closer to real data) or `tilt_max_deg=60` to `40`, re-run
6.1–6.4, and describe what changes.

---
## ✅ Checkpoint

1. A ribosome is ≈ 2.3 MDa. Using 0.81 Da/Å³, roughly how many 1-nm voxels does it occupy?
   <details><summary>Answer</summary>2.3 × 10⁶ Da ÷ 0.81 Da/Å³ ≈ 2.8 × 10⁶ Å³; one voxel is 10³ Å³ → ≈ 2,800 voxels.</details>
2. Name one advantage of single-particle cryo-EM and one of cryo-ET.
   <details><summary>Answer</summary>Single-particle: higher resolution (many identical purified copies). Cryo-ET: the cellular context — location, neighbours and states of molecules in situ.</details>
3. Why are cryo-EM images so noisy?
   <details><summary>Answer</summary>Biological samples are destroyed by electrons, so only a low total dose (roughly 20–150 electrons per Å², spread over all images) can be used; few electrons → high shot noise, low SNR.</details>
4. What is "visual proteomics"?
   <details><summary>Answer</summary>Identifying and mapping the large molecular complexes of a cell directly in 3D images (tomograms) — which molecules, how many, where, and in which states.</details>
5. Which pipeline step would you need to *discover* a complex nobody has seen before, and which modules teach it?
   <details><summary>Answer</summary>Template-free discovery / pattern mining (Modules 15 and 22), because template matching can only find known structures.</details>

## 🧠 Questions & discussion

1. Why might the *location* of a molecule inside a cell matter as much as its atomic structure? Give a biological example.
2. Our in silico cell had four kinds of molecules. List three ways a real cell is more complicated.
3. If you could only label 10 ribosomes by hand in a tomogram containing 2,000, what problems would a computer face when
   learning from them? (You will meet this in Module 31.)
4. Purification can change a complex (cofactors lost, partners removed). Which biological questions therefore *require*
   in-situ imaging?
5. Look at the pipeline table: which step do you expect to be the hardest, and why?

## 🚀 Bonus — going further

- **Explore more molecules.** The toolkit builds more stand-ins: try `zoo_map("4v7r")` (yeast 80S ribosome),
  `"5mrc"` (yeast mitochondrial ribosome), `"4cr2"` (yeast 26S proteasome), `"3cf3"` (p97) and compare their sizes.
- **Real data.** Browse the [CZ cryoET Data Portal](https://cryoetdataportal.czscience.com/) and the
  [EMPIAR archive](https://www.ebi.ac.uk/empiar/) — e.g. EMPIAR-10499 (*Mycoplasma pneumoniae* cells), used by several
  Xu-lab papers (CryoSAM, TomoPicker, DISCA).
- **Lab software.** Skim the README of [AITom](https://github.com/xulabs/aitom) and match its modules to Section 5.
- **Read** the introduction of the Kim *et al.* (2023) review with the pipeline table next to you.

## 📝 Lab note

```text
Module 00 — date:
Input: code-generated density maps (stand-ins for PDB 4V4R, 3J9I, 6UTJ, 1BXN, 1QVR, 1S3X); an in silico cell
Output: projections, mass-based contours, a simulated tomogram
Method: projection (sum), mass → volume (0.81 Da/Å³), crowding + missing wedge + CTF + noise
Parameter I changed and what happened:
A failure mode / surprise:
Paper link: Beck et al. 2011; Kim et al. 2023
```

## 📚 References

1. Beck M, Topf M, Frazier Z, Tjong H, Xu M, Zhang S, Alber F. Exploring the spatial and temporal organization of a cell's proteome. *J Struct Biol* 173:483–496 (2011). doi:10.1016/j.jsb.2010.11.011
2. Kim HH, Uddin MR, Xu M, Chang YW. Computational methods toward unbiased pattern mining and structure determination in cryo-electron tomography data. *J Mol Biol* 435:168068 (2023). doi:10.1016/j.jmb.2023.168068
3. Zeng X, Xu M. AITom: Open-source AI platform for cryo-electron tomography data analysis. arXiv:1911.03044 (2019).
4. Zhan X, Zeng X, Uddin MR, Xu M. AITom: AI-guided cryo-electron tomography image analyses toolkit. *J Struct Biol* 217:108207 (2025). doi:10.1016/j.jsb.2025.108207
5. Han R, Xu M. The advance of computational methods in cryo-electron tomography. In *Frontiers in Bioimage Informatics Methodology*, pp. 3–54 (2024).
6. Uddin MR, Dip SA, Jha RA, Zeng X, Xu M. Feature detection in cryo-electron tomography image analysis. In *Cryo-electron Tomography*, pp. 173–215, Academic Press (2025).

## ➡️ Next

**[01 · Python & NumPy for biologists: images, volumes and density maps](01_python_arrays_density_maps.ipynb)** — you
will open an MRC file byte by byte and build a density map from atoms yourself. Course overview: [`README.md`](../README.md).